# Lesson 07: Working with Python, Pandas & Scientific Text Extraction

In this notebook, you will:
1. Build index-aligned `Series` and resample time series monthly.
2. Slice, filter, transpose, and `.groupby().aggregate()` Pandas `DataFrame`s.
3. Analyze epidemiological curves and extract medical entities using open-source Python + **Google AI Studio Free Tier** (`google-genai`).

In [ ]:
!pip install -q numpy pandas matplotlib google-genai
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Series with date_range & fill_value=0
rng = np.random.default_rng(7)
idx = pd.date_range("2020-01-01", "2020-03-31")
items_sold = pd.Series(rng.integers(25, 50, size=len(idx)), index=idx)
additional_items = pd.Series(10, index=pd.date_range("2020-01-01", "2020-03-31", freq="W"))
total_items = items_sold.add(additional_items, fill_value=0)

monthly = total_items.resample("1ME").mean()
monthly.plot(kind="bar", color="#A7452A", title="Monthly Mean Ice Cream Sales")
plt.show()

In [ ]:
# DataFrame operations
a = pd.Series(range(1, 10))
b = pd.Series(["I", "like", "to", "use", "Python", "and", "Pandas", "very", "much"])
df = pd.DataFrame({"A": a, "B": b})
df["DivA"] = df["A"] - df["A"].mean()
df["LenB"] = df["B"].apply(len)
display(df.groupby("LenB").aggregate({"DivA": "count", "A": "mean"}).rename(columns={"DivA": "Count", "A": "Mean"}))

In [ ]:
# Optional: Medical abstract entity extraction with Google AI Studio Free Tier
abstract = "Randomized trial evaluating remdesivir and dexamethasone in hospitalized adults requiring supplemental oxygen."
try:
    from google.colab import userdata
    from google import genai
    api_key = userdata.get("GEMINI_API_KEY")
    if api_key:
        client = genai.Client(api_key=api_key)
        res = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=f"List medications and treatments from this abstract as JSON: {abstract}"
        )
        print(res.text)
    else:
        print("Set GEMINI_API_KEY in Colab Secrets to run live Gemini entity extraction.")
except Exception as e:
    print(f"Gemini extraction skipped: {e}")